In [64]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

spark = (
    SparkSession.builder
    .appName("Lesson21-DataFrames-SparkSQL")
    .master("local[*]")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [49]:
spark


In [65]:
orders_sch = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("amount ", DoubleType(), True),
    StructField("region", StringType(), True)
])

In [51]:

df_orders = (
    spark.read
    .option("header", "true")
    .schema(orders_sch)
    .csv("s3a://matrix/orders.csv")
)


In [61]:
df_orders = df_orders.withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-MM-dd"))

In [62]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- amount : double (nullable = true)
 |-- region: string (nullable = true)



In [54]:
!ls

Lesson17_Lab_Notebook.ipynb  final2.ipynb  shipments.json
Spark.ipynb		     less18.ipynb  spark-warehouse


In [56]:
df_shipments = spark.read.json("s3a://matrix/shipments.json")

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [46]:
df_shipments = spark.read.format("json").load("s3a://matrix/shipments.json")


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [23]:
df_shipments.printSchema()

NameError: name 'df_shipments' is not defined

In [68]:
orders_clean = (
    df_orders
    .withColumn(
        "amount ",
        F.when(F.col("amount ").isNull(), 0.0)
         .otherwise(F.col("amount "))
    )
    .withColumn(
        "amount_category",
        F.when(F.col("amount ") < 50, "small")
         .when(F.col("amount ") < 200, "medium")
         .otherwise("large")
    )
)

In [ ]:
order_ship = orders_clean.join(df_shipments,on="customer_id")

In [ ]:
orders_left_anti = orders_clean.join(df_shipments,on="order_idor", how="left_anti")

In [ ]:
orders_clean.createOrReplaceTempView("orders_clean")

orders_window = spark.sql("""
    SELECT *
    FROM (
        SELECT *,
        ROW_NUMBER() OVER (PARTITION BY customer_id  ORDER BY amount DESC NULLS LAST) AS rn
        FROM orders_clean
    ) t
    WHERE rn <= 3
""")

In [ ]:
agg_orders = spark.sql("""
        SELECT region ,
        COUNT(*) AS sifaris_say,
        SUM(amount) AS total_amount
        from orders_clean
        GROUP BY
        region 
    ORDER BY total_amount DESC
""")

In [ ]:
df_shipments.createOrReplaceTempView("df_shipments")


In [ ]:
shipment_exploded = spark.sql("""
    SELECT
         order_id,
        EXPLODE(items) AS item
    FROM df_shipments
""")

In [ ]:
shipment_exploded.createOrReplaceTempView("shipment_exploded")

In [ ]:
shipment_cal = spark.sql("""
    SELECT
        order_id,
        amount,
        SUM(line_total) AS items_total,
        SUM(line_total) - amount AS difference
    FROM shipment_exploded
     GROUP BY
        order_id,
        amount
""")

In [ ]:
customer_pivot = (
    orders_clean
    .withColumn("month", F.month("order_date"))
    .groupBy("customer_id")
    .pivot("month", [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])
    .agg(F.sum("amount"))
)

In [ ]:
cust_piv = (
    orders_clean
    .groupBy("customer_id")
    .pivot("trn_month")
    .sum("amount")
    .na.fill(0)
)

In [ ]:
(
    agg_orders.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .parquet("s3a://matrix/silver/agg_orders")
)